<a href="https://colab.research.google.com/github/zeynep-serra/ML_Notes/blob/main/R/R_NLP_DL_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
data = read.delim("reviews.tsv", quote="", stringsAsFactors=FALSE)

# install.packages('tm')
# install.packages("SnowballC")
# install.packages("randomForest")
# install.packages("caTools")
library(tm)
library(SnowballC)
library(randomForest)
library(caTools)

corpus = VCorpus(VectorSource(data$Review))
corpus = tm_map(corpus, content_transformer(tolower))
corpus = tm_map(corpus, removeNumbers)
corpus = tm_map(corpus, removePunctuation)
corpus = tm_map(corpus, removeWords, stopwords("en"))
corpus = tm_map(corpus, stemDocument)
corpus = tm_map(corpus, stripWhitespace)

# bag of words bow
dtm = DocumentTermMatrix(corpus)
dtm = removeSparseTerms(dtm, 0.999) # words seen only once

dtm_df = as.data.frame(as.matrix(dtm))

dtm_df$liked = as.factor(data$Liked)

# train test split
split = sample.split(dtm_df$liked, SplitRatio = 0.8)
training_set = subset(dtm_df, split == TRUE)
test_set = subset(dtm_df, split == FALSE)

# random forest
classifier = randomForest(x = training_set[-ncol(training_set)],
                          y = training_set$liked,
                          ntree = 500)

y_pred = predict(classifier, newdata = test_set[-ncol(test_set)])

mean(y_pred == test_set$liked)

Loading required package: NLP

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.



[1] 0.785

In [2]:
# ann / dl

data = read.csv("churn.csv")
data = data[4:14]

data$Geography = as.numeric(factor(data$Geography,
  levels = c("France", "Spain", "Germany"),
  labels = c(1, 2, 3)))

data$Gender = as.numeric(factor(data$Gender,
  levels = c("Female", "Male"),
  labels = c(0, 1)))

split = sample.split(data$Exited, SplitRatio = 0.8)
training_set = subset(data, split == TRUE)
test_set = subset(data, split == FALSE)

training_set[-11] = scale(training_set[-11])
test_set[-11] = scale(test_set[-11])

# install.packages("h2o")
library(h2o)

h2o.init(nthreads=-1)

classifier = h2o.deeplearning(y = "Exited",
  training_frame = as.h2o(training_set),
  activation = "Rectifier",
  hidden = c(6, 6),
  epochs = 100,
  train_samples_per_iteration = -2)

prob_pred = h2o.predict(classifier, newdata=as.h2o(test_set[-11]))
y_pred = (prob_pred > 0.5)
y_pred = as.vector(y_pred)
mean(y_pred == test_set[, 11])

h2o.shutdown()


----------------------------------------------------------------------

Your next step is to start H2O:
    > h2o.init()

For H2O package documentation, ask for help:
    > ??h2o

After starting H2O, you can use the Web UI at http://localhost:54321
For more information visit https://docs.h2o.ai

----------------------------------------------------------------------



Attaching package: ‘h2o’


The following objects are masked from ‘package:stats’:

    cor, sd, var


The following objects are masked from ‘package:base’:

    &&, %*%, %in%, ||, apply, as.factor, as.numeric, colnames,
    colnames<-, ifelse, is.character, is.factor, is.numeric, log,
    log10, log1p, log2, round, signif, trunc





H2O is not running yet, starting it now...

Note:  In case of errors look at the following log files:
    /tmp/RtmpNnbBVN/file3fea5e9d3552/h2o_UnknownUser_started_from_r.out
    /tmp/RtmpNnbBVN/file3fea7d8dcdd/h2o_UnknownUser_started_from_r.err


Starting H2O JVM and connecting: .... Connection successful!

R is connected to the H2O cluster: 
    H2O cluster uptime:         3 seconds 456 milliseconds 
    H2O cluster timezone:       Etc/UTC 
    H2O data parsing timezone:  UTC 
    H2O cluster version:        3.44.0.3 
    H2O cluster version age:    1 year, 9 months and 24 days 
    H2O cluster name:           H2O_started_from_R_root_tmm961 
    H2O cluster total nodes:    1 
    H2O cluster total memory:   3.17 GB 
    H2O cluster total cores:    2 
    H2O cluster allowed cores:  2 
    H2O cluster healthy:        TRUE 
    H2O Connection ip:          localhost 
    H2O Connection port:        54321 
    H2O Connection proxy:       NA 
    H2O Internal Security:      FALSE 
    R V

Warning message in h2o.clusterInfo():
“
Your H2O cluster version is (1 year, 9 months and 24 days) old. There may be a newer version available.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html”



  |======================================================================| 100%


Warning message in .h2o.processResponseWarnings(res):
“We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training..
”


  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%


[1] 0.8535

Are you sure you want to shutdown the H2O instance running at http://localhost:54321/ (Y/N)? Y
